In [ ]:
import os
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "research":
    PROJECT_ROOT = PROJECT_ROOT.parent
elif not (PROJECT_ROOT / "config" / "config.yaml").exists():
    PROJECT_ROOT = Path(r"C:\Users\vishn\Desktop\NLP\End-to-end-Text-Summarizer")

os.chdir(PROJECT_ROOT)
print(f"Project root: {Path.cwd()}")
print(f"Config exists: {(Path('config') / 'config.yaml').exists()}")
print(f"Params exists: {Path('params.yaml').exists()}")


In [ ]:
from pathlib import Path

print(f"Working directory: {Path.cwd()}")


In [ ]:
os.makedirs("artifacts/data_ingestion", exist_ok=True)


In [ ]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataValidationConfig:
    root_dir: Path
    STATUS_FILE: str
    ALL_REQUIRED_FILES: list
    unzip_data_dir: Path

In [ ]:
from textSummarizer.constants import *
from textSummarizer.utils.common import read_yaml, create_directories

In [ ]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath=CONFIG_FILE_PATH,
        params_filepath=PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])

    def get_data_validation_config(self) -> DataValidationConfig:
        config = self.config.data_validation
        create_directories([config.root_dir])

        data_validation_config = DataValidationConfig(
            root_dir=config.root_dir,
            STATUS_FILE=config.STATUS_FILE,
            ALL_REQUIRED_FILES=config.ALL_REQUIRED_FILES,
            unzip_data_dir=config.unzip_data_dir
        )

        return data_validation_config


In [ ]:
import os
from textSummarizer.logging import logger

In [ ]:
class DataValidation:
    def __init__(self, config: DataValidationConfig):
        self.config = config

    def validate_all_files_exist(self) -> bool:
        try:
            all_files = os.listdir(self.config.unzip_data_dir)

            validation_status = all(
                file in all_files
                for file in self.config.ALL_REQUIRED_FILES
            )

            with open(self.config.STATUS_FILE, "w") as f:
                f.write(f"Validation status: {validation_status}")

            return validation_status

        except Exception as e:
            raise e

In [ ]:
try:
    config = ConfigurationManager()
    data_validation_config = config.get_data_validation_config()
    data_validation = DataValidation(config=data_validation_config)
    data_validation.validate_all_files_exist()
except Exception as e:
    raise e